# Data Analysis and Visualisation

## Lab 1: Look at the data

### Preliminaries

In [ ]:
import pandas as pd

listings = pd.read_csv("data/listings.csv")

### Part 2

#### Task 1: Central or not

In [ ]:
central = ["Södermalms", "Östermalms", "Norrmalms", "Kungsholmens"]
mask = listings["neighbourhood_cleansed"].isin(central)

# Number of central listings
print(mask.sum())

# To calculate the share of listings in "central" we could divide the 
# number above by the total number of listings, like this:
print(mask.sum()/len(listings))
# But an easier way is to calculate the average of the mask, which will 
# calculate (number of True)/(total number of values)
print(mask.mean())

In [ ]:
# Share of private rooms: centre vs. outside
is_private = listings["room_type"] == "Private room"

print(is_private[mask].mean())    # inside the centre
print(is_private[~mask].mean())   # outside the centre

In [ ]:
# Alternatively, we can achieve the same with the following code:
mask2 = listings[mask]["room_type"] == "Private room"
print(mask2.mean())
mask3 = listings[~mask]["room_type"] == "Private room"
print(mask3.mean())
# Personally I think the first way is much clearer

Answer: As we can see from the results the share of private rooms is higher outside the city center.

#### Task 2: Room for a group

In [ ]:
entire_home_mask = listings["room_type"] == "Entire home/apt"
group_mask = listings["accommodates"] >= 4
print(group_mask[entire_home_mask].mean())
print(group_mask[~entire_home_mask].mean())
print(listings['room_type'].unique())

##### What does it tell us?

Nearly 56% of "Entire home" accommodations are big enough for 4 or more guests. But only 9% of accommodations of the types "Private room", "Shared room" and "Hotel room" are big enough for 4 guests. This tells us that the room type is strongly connected to the size of the listing.

##### What does it not tell us?

It does not tell me why: entire homes may also differ in price, location or number of bedrooms. The "other" group mixes private rooms, shared rooms and hotel rooms, but if we didn't know what "other" group contains, or that "room" is smaller than an "entire home / apartment", we wouldn't know why the numbers differ so much.

#### Task 3: Best reviewed, fairly

In [ ]:
many_reviews_mask = listings["number_of_reviews"] >= 50
reviewed_df = listings[many_reviews_mask]
highly_reviewed_mask = reviewed_df["review_scores_rating"] == 5.0
print(highly_reviewed_mask.sum())

The answer was 14, and that's how many listings with 50 or more reviews have ratings of exactly 5.0

In [ ]:
top = (reviewed_df[highly_reviewed_mask][["name", "neighbourhood_cleansed", "number_of_reviews", "review_scores_rating"]]
 .sort_values(["review_scores_rating", "number_of_reviews"], ascending=[False, False])
 .head(5))

top

In [ ]:
(reviewed_df[highly_reviewed_mask][["name", "neighbourhood_cleansed", "number_of_reviews", "review_scores_rating"]]
.nlargest(5, "review_scores_rating"))

The difference is due to how the ties are handled: in the case with `.sort_values()` we used a second column -- `"number_of_reviews"` -- to break the ties, while in the second we didn't.

#### Task 4: Label or position

In [ ]:
print(top.loc[347, "name"])
print(top.iloc[3]["name"])

In [ ]:
top = top.reset_index(drop=True)
display(top)
top.loc[347, "name"]


What happened: the old label 347 no longer exists, because `reset_index` replaced the labels with 0–4. Sorting and filtering keep the old labels, but `reset_index` renumbers them.

#### Task 5: What does a zero mean?

##### Part (a)

How many listings have `bedrooms` equal to `0`?

In [ ]:
no_bedrooms_mask = listings["bedrooms"] == 0
print(no_bedrooms_mask.sum())

Which `room_type` do most of them have?

In [ ]:
# To answer this we could use `.describe()`
display(listings[no_bedrooms_mask]["room_type"].describe())
# Or we could count different values and choose the top one
listings[no_bedrooms_mask]["room_type"].value_counts()

No bedrooms on a listings could mean different things. It could, for example, mean that those are studion apartments. Or, a `0` could be the default value for missing information. We don't know for sure.

##### Part (b)

How many listings have `availability_365` equal to `0`?

In [ ]:
no_availability_mask = listings["availability_365"] == 0
print(no_availability_mask.sum())

How many of these listings have `number_of_reviews` equal to `0`?

In [ ]:
no_reviews_mask = listings["number_of_reviews"] == 0
print("Listings with no availability, which also have 0 reviews:", (no_availability_mask & no_reviews_mask).sum())
print("The share of the listings with 0 reviews among the listing with no availability:", no_reviews_mask[no_availability_mask].mean())

We can see that out of 1494 listings with no availability, 231 -- or approximately 15% -- have no reviews.